In [4]:
!pip install biopython
!cp /kaggle/input/notebooks/billyjonathan/scrap-eas-text-mining/dataset_pubmed_diverse.csv /kaggle/working/dataset_pubmed_diverse.csv

In [5]:
import os
import time
import pandas as pd
from Bio import Entrez
from tqdm import tqdm

# Scrapping

In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("gmail alt")
secret_value_1 = user_secrets.get_secret("pubmed_api_key")

In [7]:
# ==========================================
# KONFIGURASI PUBMED
# ==========================================
Entrez.email = secret_value_0
Entrez.api_key = secret_value_1

# Daftar topik beragam yang ingin Anda campur di dataset
CATEGORIES = [
    "Cardiology",       # Jantung
    "Urology",          # Saluran Kemih / Ginjal
    "Neurology",        # Saraf / Otak
    "Oncology",         # Kanker / Tumor
    "Psychiatry",       # Kesehatan Mental
    "Gastroenterology", # Pencernaan
    "Immunology",       # Sistem Imun
    "Pulmonology",      # Paru-paru / Pernapasan
    "Orthopedics",      # Tulang / Otot
    "Dermatology"       # Kulit
]

PAPERS_PER_CATEGORY = 2000
BATCH_SIZE = 100
DELAY_SEC = 0.25

FILE_NAME = 'dataset_pubmed_diverse.csv'

In [8]:
pmid_sudah_ada = set()
if os.path.exists(FILE_NAME):
    df_lama = pd.read_csv(FILE_NAME)
    pmid_sudah_ada = set(df_lama['pmid'].astype(str).tolist())
    print(f"File lama ditemukan! Ada {len(pmid_sudah_ada)} artikel sebelumnya.")
else:
    print("Membuat dataset baru.")
    df_lama = pd.DataFrame()

File lama ditemukan! Ada 14492 artikel sebelumnya.


In [9]:
pmid_baru = []

print("\n=== MENGUMPULKAN PMID DARI BERBAGAI DISIPLIN ILMU ===")
for category in CATEGORIES:
    query = f'("{category}"[MeSH Terms] OR "{category}"[Title/Abstract]) AND hasabstract[text]'
    
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=PAPERS_PER_CATEGORY)
        record = Entrez.read(handle)
        handle.close()
        
        # Filter pmid yang belum ada di database lokal kita
        fetched_pmids = record["IdList"]
        unique_pmids = [p for p in fetched_pmids if p not in pmid_sudah_ada]
        
        pmid_baru.extend(unique_pmids)
        print(f"[{category.ljust(16)}] Ditemukan {len(unique_pmids)} PMID baru.")
        
        time.sleep(DELAY_SEC)
    except Exception as e:
        print(f"[{category}] Error: {e}")

pmid_baru = list(set(pmid_baru))
print(f"\nTotal PMID BARU yang siap di-scrape: {len(pmid_baru)}\n")


=== MENGUMPULKAN PMID DARI BERBAGAI DISIPLIN ILMU ===
[Cardiology      ] Ditemukan 476 PMID baru.
[Urology         ] Ditemukan 502 PMID baru.
[Neurology       ] Ditemukan 492 PMID baru.
[Oncology        ] Ditemukan 477 PMID baru.
[Psychiatry      ] Ditemukan 472 PMID baru.
[Gastroenterology] Ditemukan 499 PMID baru.
[Immunology      ] Ditemukan 475 PMID baru.
[Pulmonology     ] Ditemukan 521 PMID baru.
[Orthopedics     ] Ditemukan 511 PMID baru.
[Dermatology     ] Ditemukan 499 PMID baru.

Total PMID BARU yang siap di-scrape: 4897



In [10]:
dataset_baru = []

if len(pmid_baru) > 0:
    print("=== MULAI SCRAPING KONTEN XML ===")
    batches = [pmid_baru[i:i + BATCH_SIZE] for i in range(0, len(pmid_baru), BATCH_SIZE)]
    
    for batch in tqdm(batches, desc="Fetching Batches", unit="batch"):
        try:
            handle = Entrez.efetch(db="pubmed", id=batch, retmode="xml")
            records = Entrez.read(handle)
            handle.close()
            
            for article in records['PubmedArticle']:
                medline = article['MedlineCitation']
                pmid = str(medline['PMID'])
                
                title = medline['Article'].get('ArticleTitle', '')
                
                abstract_text = ""
                if 'Abstract' in medline['Article'] and 'AbstractText' in medline['Article']['Abstract']:
                    abstract_list = medline['Article']['Abstract']['AbstractText']
                    abstract_text = " ".join([str(txt) for txt in abstract_list])
                
                if len(abstract_text) < 50:
                    continue

                tags = []
                if 'MeshHeadingList' in medline:
                    for mesh in medline['MeshHeadingList']:
                        tags.append(str(mesh['DescriptorName']).lower())
                
                if 'KeywordList' in medline and len(medline['KeywordList']) > 0:
                    for keyword in medline['KeywordList'][0]:
                        tags.append(str(keyword).lower())
                
                tags = list(set(tags))
                
                dataset_baru.append({
                    'pmid': pmid,
                    'title': title,
                    'abstract': abstract_text,
                    'raw_tags': tags
                })
            
            time.sleep(DELAY_SEC)
            
        except Exception as e:
            print(f"\nError batch {batch[0]}-{batch[-1]}: {e}")
            time.sleep(2)

=== MULAI SCRAPING KONTEN XML ===


Fetching Batches: 100%|██████████| 49/49 [02:13<00:00,  2.72s/batch]


In [11]:
# 4. GABUNGKAN & SIMPAN (APPEND)
if len(dataset_baru) > 0:
    df_baru = pd.DataFrame(dataset_baru)
    
    # Gabungkan data lama dan data baru
    df_final = pd.concat([df_lama, df_baru], ignore_index=True)
    
    # Hapus duplikat untuk berjaga-jaga
    df_final = df_final.drop_duplicates(subset=['pmid'], keep='last')
    
    # Simpan kembali ke CSV
    df_final.to_csv(FILE_NAME, index=False)
    print(f"\nSUKSES! {len(dataset_baru)} artikel baru ditambahkan.")
    print(f"Total dataset Anda sekarang: {len(df_final)} artikel (tersimpan di {FILE_NAME})")
else:
    print("\nTidak ada artikel baru yang memenuhi syarat untuk ditambahkan.")


SUKSES! 4795 artikel baru ditambahkan.
Total dataset Anda sekarang: 19287 artikel (tersimpan di dataset_pubmed_diverse.csv)


# EDA

In [18]:
import pandas as pd
import ast
from collections import Counter

# ==========================================
# 1. LOAD DATA MENTAH PUBMED
# ==========================================
INPUT_FILE = 'dataset_pubmed_diverse.csv'
OUTPUT_FILE = 'dataset_pubmed_final.csv'

print("Membaca data...")
df = pd.read_csv(INPUT_FILE)

def safe_eval(val):
    try:
        return ast.literal_eval(val)
    except:
        return []

df['raw_tags'] = df['raw_tags'].apply(safe_eval)

# ==========================================
# 2. BLACKLIST TAGS UMUM (STOP TAGS)
# ==========================================
# Masukkan semua tag demografi, metode, dan geografi yang tidak merepresentasikan "Topik/Penyakit"
STOP_TAGS = {
    # Original Stop Tags
    'humans', 'male', 'female', 'middle aged', 'adult', 'aged', 'animals', 'retrospective studies', 'child', 'adolescent', 'treatment outcome', 
    'surveys and questionnaires', 'young adult', 'quality of life', 'cross-sectional studies', 'mice', 'risk factors', 'child, preschool', 
    'prospective studies', 'united states', 'aged, 80 and over', 'infant', 'clinical competence', 'internship and residency', 'prognosis', 'disease models, animal',
    'rats', 'cohort studies', 'time factors', 'methods', 'pandemics', 'morbidity', 'mortality', 'biomarkers', 'precision medicine', 'machine learning', 'artificial intelligence', 'pediatrics',    
    'severity of illness index', 'practice guidelines as topic', 'postoperative complications', 'prevalence', 'qualitative research', 'risk assessment', 'telemedicine', 'deep learning', 
    'case report', 'practice patterns, physicians\'', 'electronic health records', 'medical education', 'epidemiology', 'diagnosis, differential', 'reproducibility of results', 'randomized controlled trials as topic', 
    'large language models', 'confidentiality', 'tomography, x-ray computed', 'bibliometrics', 'incidence', 'consensus', 'attitude of health personnel', 'referral and consultation', 
    'biomedical research', 'algorithms', 'follow-up studies', 'hospitalization', 'curriculum', 'diagnosis', 'robotic surgical procedures', 'fibrosis', 'surgery', 'education', 'case-control studies', 
    'registries', 'quality improvement', 'chronic disease', 'bibliometric analysis', 'genetics', 'predictive value of tests', 'comorbidity', 'mutation', 'biopsy', 'disease progression', 
    'education, medical, graduate', 'treatment', 'emergency service, hospital', 'systematic review', 'obesity', 'pediatric', 'patient satisfaction', 'public health', 'ultrasonography', 'length of stay', 
    'health knowledge, attitudes, practice', 'tertiary care centers', 'primary health care', 'infectious disease', 'sex factors', 'delphi technique', 'students, medical', 'longitudinal studies', 
    'history, 20th century', 'transcriptome', 'pilot projects', 'guidelines', 'health services accessibility', 'interviews as topic', 'research design', 'phenotype', 'guideline adherence', 'patient reported outcome measures', 
    'meta-analysis', 'generative artificial intelligence', 'evidence-based medicine', 'computer security', 'critical care', 'cell differentiation', 'fellowships and scholarships', 'physicians', 'digital health', 
    'patient care team', 'clinical decision-making', 'age factors', 'educational measurement', 'risk stratification', 'proteomics', 'communication', 'feasibility studies', 'chatgpt', 'global health', 'simulation training', 
    'laparoscopy', 'genetic predisposition to disease', 'survey', 'rehabilitation', 'training', 'biomarker', 'exercise', 'metabolism', 'cell proliferation', 'patient-centered care', 'delivery of health care', 
    'patient education as topic', 'career choice', 'aging', 'safety', 'screening', 'sensitivity and specificity', 'caregivers', 'microbiome', 'health equity', 'oxidative stress', 'genetic testing', 'drug delivery systems', 
    'robotic surgery', 'pain', 'acute disease', 'single-cell analysis', 'roc curve', 'double-blind method', 'ultrasound', 'nanoparticles', 'ethics', 'medicine', 'frailty', 'paediatrics', 'trauma', 'emergency medicine', 
    'therapeutics', 'clinical decision support', 'periodicals as topic', 'regenerative medicine', 'rheumatology', 'language', 'internal medicine', 'personalized medicine', 'gene expression profiling', 
    'prevention', 'virtual reality', 'microbiota', 'magnetic resonance imaging', 'pregnancy', 'mice, inbred c57bl', 'infant, newborn', 'china', 'children', 'anti-bacterial agents', 
    'germany', 'europe', 'mice, knockout', 'united kingdom', 'italy', 'france', 'turkey', 'japan', 'primary care', 'radiology', 'physicians, women', 'disease management', 'social media', 
    'spain', 'saudi arabia', 'australia', 'india', 'brazil', 'pakistan', 'denmark', 'canada', 'history, 21st century', 'education, medical, undergraduate',
    'sweden', 'patient safety', 'east asian people', 'health personnel', 'privacy', 'ai', 'large language model', 'history, 19th century', 
    'history, 20th century', 'logistic models', 'image processing, computer-assisted', 'internet', 'management', 'multicenter studies as topic',
    'translational research, biomedical', 'dogs', 'patient acceptance of health care', 'nursing', 'social support', 'computational biology', 'vosviewer',
    'health policy', 'neural networks, computer', 'natural language processing', 'psychometrics', 'randomized controlled trial', 'complications', 
    'vaccination', 'molecular biology', 'clinical trials', 'mri', 'microbiology', 'pain management', 'drug therapy, combination', 'diagnostic imaging', 'clinical trial', 'biocompatible materials', 
    'parents', 'health sciences', 'operative time', 'mobile applications', 'diabetes', 'administration, oral', 'body mass index', 'malnutrition', 'diet', 'nutrition', 'pathology', 
    'hiv infections', 'systematic reviews as topic', 'pain measurement', 'databases, factual', 'genomics', 'transcriptomics', 'inpatients', 'dietary supplements', 'bioinformatics', 
    'infectious diseases', 'cost-benefit analysis', 'pregnancy complications', 'metabolic syndrome', 'social determinants of health', 'diabetes mellitus, type 2', 'leadership', 
    'physician-patient relations', 'outcomes', 'perioperative care', 'outpatients', 'citespace', 'general surgery', 'climate change', 'developing countries', 'imaging', 'real-world evidence', 
    'telehealth', 'health literacy', 'efficacy', 'diabetes mellitus', 'epithelial cells', 'adolescents', 'early diagnosis', 'minimally invasive surgery', 'neurosurgery', 'ambulatory care'
}

# Fungsi untuk membuang stop tags
def filter_stop_tags(tags):
    return [tag for tag in tags if tag.lower() not in STOP_TAGS]

df['clean_tags'] = df['raw_tags'].apply(filter_stop_tags)

# ==========================================
# 3. HITUNG STATISTIK TAGS BERSIH
# ==========================================
semua_tags_bersih = [tag for tags_list in df['clean_tags'] for tag in tags_list]
tag_counts = Counter(semua_tags_bersih)

MIN_FREQ = 10  

valid_tags = {tag: count for tag, count in tag_counts.items() if count >= MIN_FREQ}
sorted_valid_tags = sorted(valid_tags.items(), key=lambda x: x[1], reverse=True)

print(f"\n=== HASIL EDA MeSH TERMS (Tanpa Stop-Tags, Min {MIN_FREQ}x) ===")
print(f"Total Unique Tags Bersih : {len(sorted_valid_tags)} tags\n")

print("TOP 300 MeSH TERMS YANG RELEVAN:")
print("-" * 50)
for tag, count in sorted_valid_tags[:300]:
    print(f"{tag.ljust(50)} : {count} paper")
print("-" * 50)

Membaca data...

=== HASIL EDA MeSH TERMS (Tanpa Stop-Tags, Min 10x) ===
Total Unique Tags Bersih : 1974 tags

TOP 300 MeSH TERMS YANG RELEVAN:
--------------------------------------------------
orthopedics                                        : 845 paper
immunology                                         : 778 paper
gastroenterology                                   : 679 paper
urology                                            : 567 paper
dermatology                                        : 566 paper
neurology                                          : 496 paper
neoplasms                                          : 467 paper
pulmonology                                        : 439 paper
psychiatry                                         : 425 paper
cardiology                                         : 402 paper
inflammation                                       : 371 paper
covid-19                                           : 364 paper
oncology                                         

# Tag Mapping

In [19]:
# ==========================================
# 4. KAMUS GENERALISASI (Di-update dengan Top 200 Terbaru)
# ==========================================
tag_mapping = {
    'Cardiology': [
        'cardiology', 'heart diseases', 'myocardial infarction', 'hypertension', 'blood pressure', 
        'echocardiography', 'heart failure', 'atrial fibrillation', 'cardiovascular diseases', 
        'cardiovascular disease', 'coronary artery disease', 'acute coronary syndrome', 
        'congenital heart disease', 'interventional cardiology', 'electrocardiography', 
        'cardio-oncology', 'percutaneous coronary intervention', 'stents', 'anticoagulants', 
        'cardiotoxicity', 'atherosclerosis', 'heart defects, congenital', 'pulmonary embolism',
        'coronary angiography', 'vascular biology'
    ],
    'Urology': [
        'urology', 'kidney diseases', 'urologic diseases', 'prostatic neoplasms', 'urinary tract infections', 
        'pediatric urology', 'prostate cancer', 'urinary bladder neoplasms', 'urologic surgical procedures', 
        'epilepsy', 'prostatectomy', 'kidney calculi', 'bladder cancer', 'urolithiasis', 'prostate', 
        'lower urinary tract symptoms', 'kidney', 'urologists', 'prostatic hyperplasia', 
        'kidney neoplasms', 'ureteroscopy', 'ureter', 'nephrectomy', 'urinary bladder', 
        'erectile dysfunction', 'ureteral obstruction', 'nephrology'
    ],
    'Neurology': [
        'neurology', 'brain diseases', 'stroke', 'alzheimer disease', 'parkinson disease', 
        'nervous system diseases', 'multiple sclerosis', 'epilepsy', 'brain', 'neuroimaging', 
        'neuroscience', 'ischemic stroke', 'dementia', 'pediatric neurology', 'clinical neurology', 
        'neuroinflammation', 'cognitive dysfunction', 'cognition', 'electroencephalography', 
        'brain neoplasms', 'neurologists', 'glioblastoma', 'seizures', 'anticonvulsants', 'chronic pain'
    ],
    'Oncology': [
        'neoplasms', 'oncology', 'cancer', 'immunotherapy', 'tumor microenvironment', 'lung neoplasms', 
        'skin neoplasms', 'prostatic neoplasms', 'prostate cancer', 'antineoplastic agents', 
        'palliative care', 'lung cancer', 'cell line, tumor', 'biomarkers, tumor', 
        'immune checkpoint inhibitors', 'breast cancer', 'urinary bladder neoplasms', 'precision oncology', 
        'colorectal neoplasms', 'medical oncology', 'melanoma', 'early detection of cancer', 
        'cancer immunotherapy', 'cardio-oncology', 'breast neoplasms', 'neoplasm staging', 
        'bladder cancer', 'targeted therapy', 'antineoplastic combined chemotherapy protocols', 
        'gene expression regulation, neoplastic', 'neoplasm recurrence, local', 'apoptosis', 
        'chemotherapy', 'skin cancer', 'combined modality therapy', 'tumor immunology', 
        'cardiotoxicity', 'kidney neoplasms', 'liver neoplasms', 'liquid biopsy', 'brain neoplasms', 
        'pancreatic neoplasms', 'glioblastoma', 'stomach neoplasms', 'positron-emission tomography',
        'hepatocellular carcinoma', 'molecular targeted therapy', 'neoplasm invasiveness', 
        'dna methylation', 'pediatric oncology', 'cancer therapy', 'neoplasm metastasis'
    ],
    'Psychiatry': [
        'psychiatry', 'depression', 'mental disorders', 'mental health', 'anxiety', 'schizophrenia', 
        'major depressive disorder', 'mental health services', 'forensic psychiatry', 'bipolar disorder', 
        'precision psychiatry', 'cognitive dysfunction', 'cognition', 'anxiety disorders', 
        'stress, psychological', 'child psychiatry', 'child and adolescent psychiatry', 
        'antipsychotic agents', 'sleep quality', 'antidepressive agents', 'suicide', 'psychotherapy', 
        'adaptation, psychological', 'chronic pain', 'burnout, professional'
    ],
    'Gastroenterology': [
        'gastroenterology', 'inflammatory bowel disease', 'gastrointestinal microbiome', 
        'gastrointestinal diseases', 'colonoscopy', 'colorectal neoplasms', 'ulcerative colitis', 
        'crohn disease', 'hepatology', 'helicobacter pylori', 'liver diseases', 'liver', 
        'endoscopy, gastrointestinal', 'liver cirrhosis', 'colitis, ulcerative', 'crohn’s disease', 
        'probiotics', 'abdominal pain', 'celiac disease', 'gut microbiota', 'gastroenterologists', 
        'liver neoplasms', 'intestinal mucosa', 'pancreatic neoplasms', 'stomach neoplasms', 
        'irritable bowel syndrome', 'dysbiosis', 'helicobacter infections', 'constipation', 
        'hepatocellular carcinoma', 'liver transplantation'
    ],
    'Immunology': [
        'immunology', 'inflammation', 'immunotherapy', 'tumor microenvironment', 'cp: immunology', 
        'cytokines', 'macrophages', 'innate immunity', 'immunity, innate', 'antibodies, monoclonal, humanized', 
        'autoimmune diseases', 'autoimmunity', 'cancer immunotherapy', 't-lymphocytes', 
        'cd8-positive t-lymphocytes', 'neutrophils', 't cells', 'autoantibodies', 'flow cytometry', 
        'fibroblasts', 'infection', 'sepsis', 'apoptosis', 'epigenetics', 'exosomes', 'mitochondria', 
        'tumor immunology', 'biologics', 'extracellular vesicles', 'stem cells', 'genotype', 
        'membrane proteins', 'antibodies, monoclonal', 'adaptive immunity', 'biological products', 
        'mesenchymal stem cells', 'allergy', 'dendritic cells', 'microbial sensitivity tests', 
        'dna methylation', 't-lymphocytes, regulatory', 'immunosuppressive agents', 'neuroinflammation'
    ],
    'Pulmonology': [
        'pulmonology', 'covid-19', 'lung', 'asthma', 'sars-cov-2', 'bronchoscopy', 'pulmonary medicine', 
        'lung neoplasms', 'interventional pulmonology', 'lung cancer', 'pulmonary disease, chronic obstructive', 
        'cystic fibrosis', 'lung diseases, interstitial', 'lung diseases', 'tuberculosis', 
        'idiopathic pulmonary fibrosis', 'copd', 'pneumonia', 'interstitial lung disease', 
        'lung transplantation/pulmonology', 'chronic obstructive pulmonary disease', 'pulmonary fibrosis', 
        'dyspnea', 'pulmonary embolism', 'allergy', 'pediatric pulmonology', 'respiratory distress syndrome'
    ],
    'Orthopedics': [
        'orthopedics', 'orthopedic procedures', 'pediatric orthopedics', 'orthopedic surgeons', 
        'traumatology', 'orthopedic surgery', 'bone regeneration', 'tissue engineering', 
        'osteogenesis', 'osteoporosis', 'orthopaedics', 'arthroplasty, replacement, knee', 
        'arthroplasty, replacement, hip', 'fractures, bone', 'fracture fixation, internal', 
        'musculoskeletal diseases', 'platelet-rich plasma'
    ],
    'Dermatology': [
        'dermatology', 'skin', 'skin diseases', 'atopic dermatitis', 'skin neoplasms', 'melanoma', 
        'dermatitis, atopic', 'pediatric dermatology', 'dermatologic agents', 'hidradenitis suppurativa', 
        'dermoscopy', 'skin cancer', 'skin aging', 'acne vulgaris', 'wound healing', 'platelet-rich plasma', 
        'pruritus'
    ]
}

# ==========================================
# 5. TERAPKAN MAPPING & FILTERING
# ==========================================
def map_tags_to_categories(tags_list):
    mapped_categories = set()
    for tag in tags_list:
        tag_lower = tag.lower()
        for category, keywords in tag_mapping.items():
            if tag_lower in keywords: 
                mapped_categories.add(category)
    return list(mapped_categories)

df['final_labels'] = df['clean_tags'].apply(map_tags_to_categories)

# Filter: Hanya simpan paper yang punya minimal 1 final_label
jumlah_awal = len(df)
df_final = df[df['final_labels'].map(len) > 0].copy()
jumlah_akhir = len(df_final)

# ==========================================
# 6. SIMPAN & REPORT FINAL
# ==========================================
df_final.to_csv(OUTPUT_FILE, index=False)

print("\n=== REPORT DATASET FINAL PUBMED ===")
print(f"Total Paper Awal  : {jumlah_awal}")
print(f"Total Paper Final : {jumlah_akhir} ({(jumlah_akhir/jumlah_awal)*100:.2f}% terselamatkan)\n")

print("Distribusi Kategori Label Utama pada Dataset Final:")
all_final_labels = [label for labels in df_final['final_labels'] for label in labels]
label_counts = Counter(all_final_labels)

for label, count in label_counts.most_common():
    print(f"- {label.ljust(20)} : {count} paper")
print(f"\nDataset bersih tersimpan sebagai: '{OUTPUT_FILE}'")


=== REPORT DATASET FINAL PUBMED ===
Total Paper Awal  : 19287
Total Paper Final : 12479 (64.70% terselamatkan)

Distribusi Kategori Label Utama pada Dataset Final:
- Oncology             : 2734 paper
- Immunology           : 2514 paper
- Pulmonology          : 1827 paper
- Psychiatry           : 1532 paper
- Gastroenterology     : 1508 paper
- Cardiology           : 1482 paper
- Neurology            : 1461 paper
- Urology              : 1405 paper
- Dermatology          : 1220 paper
- Orthopedics          : 1218 paper

Dataset bersih tersimpan sebagai: 'dataset_pubmed_final.csv'
